In [1]:
# [Cell 2] Install LangChain, Hugging Face integrations, and tools
!pip install -qU transformers accelerate bitsandbytes 
!pip install -qU langchain langchain-core langchain-community langchain-huggingface
!pip install -qU wikipedia duckduckgo-search
!pip install -U ddgs


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [2]:
import torch

print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"ROCm is active! Found {torch.cuda.device_count()} AMD GPU(s).")
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("Warning: ROCm/CUDA not available. Check your AMD Developer Cloud instance.")

PyTorch version: 2.8.0+gitb2fb688
ROCm is active! Found 1 AMD GPU(s).
Device Name: 


In [3]:
# [Cell 3] Load the LLM and wrap it for Chat/Tool compatibility
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

model_id = "Qwen/Qwen2-7B-Instruct" 

print(f"Loading tokenizer and model {model_id} onto AMD GPU...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",    # Automatically maps to AMD ROCm
    torch_dtype=torch.bfloat16 # Optimized for Instinct GPUs
)

# Create a text generation pipeline
text_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    max_length=None,
    temperature=0.1,
    return_full_text=False
)

# Wrap it in LangChain's pipeline, then into the modern Chat wrapper
llm = HuggingFacePipeline(pipeline=text_pipeline)
chat_model = ChatHuggingFace(llm=llm, model_id=model_id)

print("Qwen Chat Model Ready!")

Loading tokenizer and model Qwen/Qwen2-7B-Instruct onto AMD GPU...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_length', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Qwen Chat Model Ready!


In [4]:
chat_model.invoke("Hi")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


AIMessage(content='Hello! How can I assist you today?', additional_kwargs={}, response_metadata={}, id='lc_run--019ed0c8-dfa9-7622-9fb3-ef79717cdf7a-0', tool_calls=[], invalid_tool_calls=[])

In [5]:
# [Cell 4] Set up the agent's toolkit (Fixed for Python 3.12+)
import uuid # Safety net for dynamic type evaluation
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun

# Initialize the tools directly as native LangChain Tool objects
search_tool = DuckDuckGoSearchRun()
wikipedia_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

# Pass the pre-built tools directly instead of wrapping them
tools = [search_tool, wikipedia_tool]

print(f"Loaded {len(tools)} tools: {[t.name for t in tools]}")

Loaded 2 tools: ['duckduckgo_search', 'wikipedia']


In [6]:
# [Cell 5] Initialize and test the Agent using LangChain v1.0+
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# Create the modern agent
agent = create_agent(
    model=chat_model,
    tools=tools,
    system_prompt="You are an advanced AI assistant powered by Qwen2 and running on an AMD MI300X GPU. Use your tools when necessary to provide highly accurate, factual answers."
)

# Test the Agent!
query = "What is the architecture of the AMD Instinct MI300X, and how does it benefit AI agents?"

print(f"User Query: {query}\n" + "="*50)

# Invoke the agent using the standard messages list
response = agent.invoke({
    "messages": [HumanMessage(content=query)]
})

print("="*50)
# The final response is the content of the last message in the state
print(f"Agent Final Answer: {response['messages'][-1].content}")

User Query: What is the architecture of the AMD Instinct MI300X, and how does it benefit AI agents?
Agent Final Answer: The AMD Instinct MI300X is a high-performance accelerator card designed specifically for AI and HPC (High-Performance Computing) applications. It leverages the AMD CDNA 2 architecture, which is optimized for compute-intensive workloads, including machine learning training and inference, scientific simulations, and data analytics.

### Key Features of AMD Instinct MI300X Architecture:

1. **CDNA 2 Architecture**: This architecture is designed with a focus on delivering high performance in AI and HPC tasks. It includes features like:
   - **Compute Cores**: The MI300X has a large number of compute cores that can execute parallel tasks efficiently.
   - **Memory Bandwidth**: It supports high memory bandwidth to handle large datasets quickly, crucial for AI training where vast amounts of data need to be processed.

2. **Memory System**:
   - **HBM3 Memory**: The MI300X ut